<div align="left" style="background-color: #008080; padding: 20px 10px;">
<h3><b>IDEAS - Institute of Data Engineering, Analytics and Science Foundation</b></h3>
<p>Summer Internship Program 2026</p>
<hr style="width:100%;">
<h3><b>Project Title:</b> Anomaly Detection for Fraud and Sensor Data</h3>
<h4>Project Notebook</h4>

<blockquote style="border-left: 4px solid #4285F4; padding-left: 15px;">
  <strong>Created by:</strong> Rounak Biswas<br>
  <strong>Designation:</strong> Project Linked Associate Research Engineer
</blockquote>
<hr style="width:100%;">
</div>

### Question 1: Load Libraries (2 Marks)

Import `numpy` as `np`, `pandas` as `pd`, `stats` from `scipy`, `IsolationForest` and `LocalOutlierFactor` from `sklearn.ensemble` and `sklearn.neighbors`, and `classification_report`, `precision_score`, `recall_score` from `sklearn.metrics`.

**Expected Output:** The code cell should execute without any errors.

In [ ]:
# Write your answer here
import numpy as np
import pandas as pd

from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    classification_report,
    precision_score,
    recall_score
)

### Question 2: Create the Dataset (4 Marks)

Generate a synthetic credit card transaction dataset. Set `np.random.seed(42)`. Create a DataFrame `normal` with 950 rows and a DataFrame `fraud` with 50 rows. Features should include `amount`, `hour_of_day`, `transactions_last_24h`, and `distance_from_home_km`, plus an `is_fraud` label (0 for normal, 1 for fraud). Combine them into a single DataFrame named `df`, shuffle using `.sample(frac=1, random_state=42)`, and reset the index.

**Hint:** Use `np.random.normal`, `np.random.randint`, `np.random.poisson`, and `np.random.exponential` to generate feature data.

**Expected Output:** Execution without errors, creating the `df` DataFrame.

In [ ]:
# Write your answer here
np.random.seed(42)

normal = pd.DataFrame({
    'amount': np.random.normal(100, 30, 950),
    'hour_of_day': np.random.randint(6, 23, 950),
    'transactions_last_24h': np.random.poisson(3, 950),
    'distance_from_home_km': np.random.exponential(5, 950),
    'is_fraud': 0
})

fraud = pd.DataFrame({
    'amount': np.random.normal(500, 150, 50),
    'hour_of_day': np.random.randint(0, 24, 50),
    'transactions_last_24h': np.random.poisson(10, 50),
    'distance_from_home_km': np.random.exponential(30, 50),
    'is_fraud': 1
})

df = pd.concat([normal, fraud])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

### Question 3: Check Dataset Shape and Distribution (2 Marks)

Print the shape of your combined DataFrame `df` and the value counts of the `is_fraud` column to observe the class distribution.

**Expected Output:** The shape (1000, 5) and the counts showing 950 normal (0) and 50 fraud (1) cases.

In [ ]:
# Write your answer here
print(df.shape)
print(df['is_fraud'].value_counts())

(1000, 5)
is_fraud
0    950
1     50
Name: count, dtype: int64


### Question 4: Compare Feature Means by Class (3 Marks)

Group the dataset by the `is_fraud` column and calculate the mean values for all features (`amount`, `hour_of_day`, `transactions_last_24h`, `distance_from_home_km`). Print the resulting grouped means.

**Hint:** Use the `.groupby()` method and `.mean()`.

**Expected Output:** A table showing the mean feature values for normal (0) vs fraudulent (1) transactions.

In [ ]:
# Write your answer here
print(df.groupby('is_fraud').mean())

              amount  hour_of_day  transactions_last_24h  \
is_fraud                                                   
0         100.602522    13.965263               2.962105   
1         481.467231    11.180000               9.980000   

          distance_from_home_km  
is_fraud                         
0                      5.035512  
1                     26.567519  


### Question 5: Apply Z-Score Anomaly Detection (3 Marks)

Compute the absolute Z-scores for the `distance_from_home_km` column. Create a new column named `zscore_anomaly` in `df` that contains `1` if the absolute Z-score is greater than 3, and `0` otherwise. Print the total number of flagged anomalies.

**Hint:** Use `np.abs(stats.zscore(...))`.

**Expected Output:** The count of anomalies flagged based on the distance feature.

In [ ]:
# Write your answer here
z_scores = np.abs(stats.zscore(df['distance_from_home_km']))
df['zscore_anomaly'] = (z_scores > 3).astype(int)
print(df['zscore_anomaly'].sum())

18


### Question 6: Apply IQR Method (4 Marks)

Calculate the Interquartile Range (IQR) for the `amount` column. Identify bounds: `Lower = Q1 - 1.5 * IQR` and `Upper = Q3 + 1.5 * IQR`. Create a new column named `iqr_anomaly` in `df` containing `1` for values outside these bounds and `0` otherwise. Print the total number of flagged anomalies.

**Hint:** Use `.quantile(0.25)` for Q1 and `.quantile(0.75)` for Q3.

**Expected Output:** The total number of `amount` anomalies flagged by the IQR method.

In [ ]:
# Write your answer here
Q1 = df['amount'].quantile(0.25)
Q3 = df['amount'].quantile(0.75)

IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['iqr_anomaly'] = (
    (df['amount'] < lower_bound) |
    (df['amount'] > upper_bound)
).astype(int)

print(df['iqr_anomaly'].sum())

54


### Question 7: Train Isolation Forest (4 Marks)

Import `StandardScaler` from `sklearn.preprocessing` and scale the four feature columns. Then, create an `IsolationForest` model with `contamination=0.05` and `random_state=42`. Fit the model on the scaled features and add a column `isoforest_anomaly` to `df` containing `1` for anomalies and `0` for normal data.

**Hint:** `IsolationForest` returns `-1` for anomalies and `1` for normal points. Map these to `1` and `0` respectively.

**Expected Output:** The execution completes successfully.

In [ ]:
# Write your answer here
from sklearn.preprocessing import StandardScaler

features = ['amount', 'hour_of_day', 'transactions_last_24h', 'distance_from_home_km']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features])

iso_forest = IsolationForest(
    contamination=0.05,
    random_state=42
)

predictions = iso_forest.fit_predict(X_scaled)

df['isoforest_anomaly'] = (predictions == -1).astype(int)

### Question 8: Evaluate Isolation Forest (3 Marks)

Use the `classification_report` function to evaluate the performance of your `isoforest_anomaly` predictions against the true `is_fraud` labels. Print the report.

**Expected Output:** A classification report displaying precision, recall, and f1-score for the model.

In [ ]:
# Write your answer here
print(classification_report(
    df['is_fraud'],
    df['isoforest_anomaly']
))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       950
           1       0.92      0.92      0.92        50

    accuracy                           0.99      1000
   macro avg       0.96      0.96      0.96      1000
weighted avg       0.99      0.99      0.99      1000



### Question 9: Local Outlier Factor Detection (3 Marks)

Train a `LocalOutlierFactor` model with `n_neighbors=20` and `contamination=0.05` on the scaled features. Add a new column `lof_anomaly` to `df` (where `1` indicates an anomaly and `0` indicates normal).

**Hint:** Use `.fit_predict()` to get the anomaly flags (similar to Isolation Forest, LOF returns `-1` for anomalies).

**Expected Output:** The execution completes successfully.

In [ ]:
# Write your answer here
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.05
)

lof_predictions = lof.fit_predict(X_scaled)

df['lof_anomaly'] = (lof_predictions == -1).astype(int)

### Question 10: Evaluate LOF Model (2 Marks)

Calculate and print both the `precision_score` and `recall_score` for the `lof_anomaly` predictions against the true `is_fraud` labels.

**Expected Output:** Two numbers showing the precision and recall scores.

In [ ]:
# Write your answer here
precision = precision_score(
    df['is_fraud'],
    df['lof_anomaly']
)
recall = recall_score(
    df['is_fraud'],
    df['lof_anomaly']
)
print("Precision:", precision)
print("Recall:", recall)

Precision: 0.36
Recall: 0.36
